# 01｜坐标系与齐次变换交互实验

对应[第 01 章](../course/01-coordinates-and-kinematics.md)。本 Notebook 用可视化建立直觉；完成后仍需独立补全 [`01_transform_3d_exercise.py`](../labs/starter/01_transform_3d_exercise.py)。

## 学习目标

完成后你应能：区分点与方向；读懂 $T^A_B$；按坐标链组合变换；解释变换为何不可交换；把结论连接到 Panda 的 world/base/tip 坐标系。

## 本节知识地图

本节只学 5 个知识点。先认清名字，再进入预测。

| 知识点 | 一句话解释 | 在 Panda 中对应什么 | 掌握检查 |
| --- | --- | --- | --- |
| 点与方向 | 点有位置；方向的齐次末位为 0，所以不受平移 | 方块位置 vs 运动方向 | 能解释平移为何不改变方向 |
| 坐标系与表达 | 同一个点换坐标系后数字会变 | world/base/tip/camera | 每个坐标都能说出表达系 |
| 旋转矩阵 $R$ | 改变方向表达，不包含平移 | 末端和相机姿态 | 能检查 $R^T R=I$ |
| 齐次变换 $T$ | 4×4 矩阵统一旋转和平移 | `world_from_base` | 手算一个点的变换 |
| 坐标链 | 相邻表达系按路径相乘，顺序不可交换 | world ← base ← tip | 写对矩阵乘法顺序 |

## 关键概念与符号

| 符号 | 读法 | 含义 |
| --- | --- | --- |
| ${}^B p$ | 在 B 系表达的点 p | 输入坐标，不是另一个点 |
| ${}^A R_B$ | B 到 A 的旋转 | 把 B 表达的方向改写为 A 表达 |
| ${}^A t_B$ | B 原点在 A 系的位置 | 平移向量 |
| ${}^A T_B$ | B 到 A 的齐次变换 | 把 B 表达的点变成 A 表达 |

> **不要混淆：** 几何对象本身没有移动，也可能只是换了坐标表达；矩阵下标说明“从哪里到哪里”，代码变量名 `world_from_base` 说明同一件事。

## 先预测

先不要运行代码，在纸上回答：

1. B 系相对 world 逆时针旋转 90°并平移 `[2, 1, 0]`，B 中点 `[1, 0, 0]` 在 world 中是什么？
2. `T_world_base @ T_base_tip` 与反序结果会相同吗？
3. 齐次末位为 0 的方向会不会受到平移？

把预测写进 `notes/01-notebook-prediction.md`，再运行下面的 cell。

## 运行与观察

第一段只建立项目路径并检查 kernel。若这里提示 Wrong kernel，请关闭当前 Jupyter，回到终端运行 `./reproduction/start_course_notebooks.sh`。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
from course_utils import assert_course_kernel, rigid_transform, rotation_z, transform_point

assert_course_kernel(ROOT)
print('Repository:', ROOT)
print('Kernel:', sys.executable)

### 1. 点经过旋转和平移

矩阵 $T^A_B$ 把 B 系表达的点变成 A 系表达。右边最先作用：`translation @ rotation @ point` 表示先旋转点，再平移。

In [ ]:
point_b = np.array([1.0, 0.0, 0.0])
world_from_b = rigid_transform(rotation_z(90.0), np.array([2.0, 1.0, 0.0]))
point_world = transform_point(world_from_b, point_b)

print('B coordinates:    ', point_b)
print('World coordinates:', np.round(point_world, 6))
assert np.allclose(point_world, [2.0, 2.0, 0.0])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color='0.85'); ax.axvline(0, color='0.85')
ax.quiver(0, 0, 1, 0, angles='xy', scale_units='xy', scale=1, color='tab:red', label='world x')
ax.quiver(0, 0, 0, 1, angles='xy', scale_units='xy', scale=1, color='tab:green', label='world y')
origin_b = world_from_b[:2, 3]
axes_b = world_from_b[:2, :2]
ax.quiver(*origin_b, *axes_b[:, 0], angles='xy', scale_units='xy', scale=1, color='darkred')
ax.quiver(*origin_b, *axes_b[:, 1], angles='xy', scale_units='xy', scale=1, color='darkgreen')
ax.scatter(*point_world[:2], s=80, color='tab:blue', label='same physical point')
ax.set(xlim=(-0.5, 3.5), ylim=(-0.5, 3.5), aspect='equal', xlabel='world x', ylabel='world y')
ax.legend(); ax.set_title('World frame and rotated/translated B frame')
plt.show()

### 2. 组合顺序不可交换

下面用同一个点比较“先旋转后平移”和“先平移后旋转”。请先预测两行输出，再运行。

In [ ]:
rotation = rigid_transform(rotation_z(90.0), np.zeros(3))
translation = rigid_transform(np.eye(3), np.array([2.0, 1.0, 0.0]))
rotate_then_translate = transform_point(translation @ rotation, point_b)
translate_then_rotate = transform_point(rotation @ translation, point_b)
print('rotate → translate:', np.round(rotate_then_translate, 6))
print('translate → rotate:', np.round(translate_then_rotate, 6))
assert np.allclose(rotate_then_translate, [2.0, 2.0, 0.0])
assert np.allclose(translate_then_rotate, [-1.0, 3.0, 0.0])

### 3. world ← base ← tip 坐标链

把 tip 原点看成一个点。`world_from_base @ base_from_tip` 中间的 base 语义相消，输出在 world 中表达。

In [ ]:
base_from_tip = rigid_transform(np.eye(3), np.array([1.0, 0.0, 0.25]))
world_from_tip = world_from_b @ base_from_tip
tip_world = transform_point(world_from_tip, np.zeros(3))
print('Tip origin in world:', np.round(tip_world, 6))
assert np.allclose(tip_world, [2.0, 2.0, 0.25])
assert np.allclose(world_from_tip @ np.linalg.inv(world_from_tip), np.eye(4))

## 动手修改

把 `angle` 依次改成 0、45、180；每次先画出点的大致象限，再运行。观察旋转改变的是点相对 B 原点的方向，而平移决定 B 原点在 world 的位置。完成后恢复 45，保证后面的自测具有确定输入。

In [ ]:
angle = 45.0  # 修改这一行做实验，最后恢复为 45.0
trial_transform = rigid_transform(rotation_z(angle), np.array([2.0, 1.0, 0.0]))
trial_point = transform_point(trial_transform, point_b)
print(f'angle={angle:.1f}° -> world point={np.round(trial_point, 4)}')

## 自测

运行断言前，口头解释每一条在检查什么。预期最后打印 `PASS`。

In [ ]:
r = rotation_z(45.0)
assert np.allclose(r.T @ r, np.eye(3), atol=1e-9)
assert np.isclose(np.linalg.det(r), 1.0)
assert np.allclose(trial_point, [2 + np.sqrt(0.5), 1 + np.sqrt(0.5), 0])
direction = np.array([1.0, 0.0, 0.0, 0.0])
assert np.allclose((trial_transform @ direction)[:3], r @ direction[:3])
print('PASS: rotation validity, point transform, and direction semantics')

## 学完请记住

关闭本页后，你应能脱稿说出：

1. 点的齐次末位为 1，方向为 0，所以只有点受到平移；
2. ${}^A T_B$ 把 B 系坐标改写成 A 系坐标；
3. `world_from_base @ base_from_tip` 的中间 base 语义相接；
4. 右侧矩阵先作用，旋转和平移通常不可交换；
5. Panda 的笛卡尔目标最终由 IK 变成关节控制。

若第 2 或第 3 条说不清，只回看上面的符号表和“world ← base ← tip”，不必重读整章。

## 反思与记录

在 `notes/01-notebook-reflection.md` 回答：

1. 为什么矩阵代码从右向左体现动作顺序？
2. 为什么相机画面左侧不能直接叫 world 负 y？
3. Panda 策略输出 y/z 增量后，在哪一步从笛卡尔目标变成关节控制？

随后关闭答案，独立完成三维 starter。Notebook 自测通过只证明你理解了示例，不等于 Gate 1 READY。